In [1]:
# Memory safety: cap this kernel to the RAM free right now so an out-of-memory
# feature build fails with a clean MemoryError instead of crashing VS Code /
# thrashing swap. This notebook builds cross-row features (lags/rolling), which
# can't be row-batched, so the guard is the protection here.
import os, sys
sys.path.insert(0, os.path.abspath("../../../Generic-Parallel-Compute-Helper/")) ; from memory_compute import *
install_memory_guard()


[memory_guard] hard cap 13.3G virtual on this kernel (total RAM 14.8G, 9.3G free now). Runaway allocations fail cleanly; bounded streaming keeps normal work well under this.


14298148864

In [2]:
import os
import sys
import numpy as np
import pandas as pd

sys.path.append(os.path.abspath("../../")) ; from EPF import variables

# All time windows below are expressed against the native 5-minute feature
# index. Keep the conversion in one place so changing feature granularity
# cannot silently turn days/weeks/years into much shorter windows.
PER_HOUR = 60 // variables.FEATURE_GRANULARITY_IN_MINUTES
PER_DAY = 24 * PER_HOUR
PER_WEEK = 7 * PER_DAY
PER_YEAR = int(round(365.25 * PER_DAY))
HALF_HOUR = max(1, 30 // variables.FEATURE_GRANULARITY_IN_MINUTES)

In [3]:
df = read_parquet_float32("../1_Dataset/Processed_data/1_dispatch_price.parquet")

df_core_columns = df.columns
df_base = df
df_base[:10]

Loading..: 100%|██████████| 9/9 [00:00<00:00, 83.40batch/s]


,nsw_price,qld_price,sa_price,vic_price
Date,,,,
2018-01-01 00:05:00,90.209999,80.270287,103.578568,90.609642
2018-01-01 00:10:00,95.540657,85.100067,111.817146,96.736847
2018-01-01 00:15:00,96.293869,85.099998,113.702904,97.344391
2018-01-01 00:20:00,92.500000,81.728340,106.751778,92.824448
2018-01-01 00:25:00,92.499901,81.708931,108.531883,92.833130
2018-01-01 00:30:00,84.092339,73.730026,98.659790,84.389008
2018-01-01 00:35:00,92.449867,81.088654,105.000053,91.349518
2018-01-01 00:40:00,95.790154,84.599960,109.102798,93.914917
2018-01-01 00:45:00,90.997910,79.500816,105.000053,89.812172


In [4]:
def _add_arcsinh_price_lags(df: pd.DataFrame) -> pd.DataFrame:
    """
    arcsinh-transformed price lags.
    arcsinh handles negatives and compresses extreme spikes,
    giving the model a better-scaled view of price history.
    Returns only the new columns to avoid copying the full base frame.
    """
    scale = 100
    new_cols = {}
    for col in df_core_columns:
        ap = np.arcsinh(df[col] / scale).rename("_ap")
        for lag in [
            1, 2, 4, HALF_HOUR, PER_HOUR,
            PER_DAY, 2 * PER_DAY,
            PER_WEEK - HALF_HOUR, PER_WEEK, PER_WEEK + HALF_HOUR,
        ]:
            new_cols[f"{col}_asinh_lag_{lag}"] = ap.shift(lag).astype(np.float32)
        new_cols[f"{col}_asinh_rmean_{PER_DAY}"] = ap.rolling(PER_DAY).mean().astype(np.float32)
        new_cols[f"{col}_asinh_rmean_{PER_WEEK}"] = ap.rolling(PER_WEEK).mean().astype(np.float32)
    return pd.DataFrame(new_cols, index=df.index)

new_df = _add_arcsinh_price_lags(df_base)

df = pd.concat([df, new_df], axis=1)

new_df[:10]



,nsw_price_asinh_lag_1,nsw_price_asinh_lag_2,nsw_price_asinh_lag_4,nsw_price_asinh_lag_6,nsw_price_asinh_lag_12,nsw_price_asinh_lag_288,nsw_price_asinh_lag_576,nsw_price_asinh_lag_2010,nsw_price_asinh_lag_2016,nsw_price_asinh_lag_2022,...,vic_price_asinh_lag_4,vic_price_asinh_lag_6,vic_price_asinh_lag_12,vic_price_asinh_lag_288,vic_price_asinh_lag_576,vic_price_asinh_lag_2010,vic_price_asinh_lag_2016,vic_price_asinh_lag_2022,vic_price_asinh_rmean_288,vic_price_asinh_rmean_2016
Date,,,,,,,,,,,,,,,,,,,,,
2018-01-01 00:05:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:10:00,0.810427,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:15:00,0.849487,0.810427,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:20:00,0.854923,0.849487,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:25:00,0.827334,0.854923,0.810427,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.813392,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:30:00,0.827333,0.827334,0.849487,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.858110,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:35:00,0.764306,0.827333,0.854923,0.810427,NaN,NaN,NaN,NaN,NaN,NaN,...,0.862470,0.813392,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:40:00,0.826966,0.764306,0.827334,0.849487,NaN,NaN,NaN,NaN,NaN,NaN,...,0.829714,0.858110,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:45:00,0.851290,0.826966,0.827333,0.854923,NaN,NaN,NaN,NaN,NaN,NaN,...,0.829778,0.862470,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
def _add_time_features(df: pd.DataFrame) -> pd.DataFrame:

    import holidays

    idx = df.index
    region_holidays = holidays.Australia(
        state=variables.TARGET_REGION.upper(),
        years=range(int(idx.year.min()), int(idx.year.max()) + 1),
    )
    new_cols = {}

    # Raw calendar components
    new_cols["hour"]        = idx.hour.astype(np.int8)
    new_cols["dayofweek"]   = idx.dayofweek.astype(np.int8)
    new_cols["month"]       = idx.month.astype(np.int8)
    new_cols["dayofyear"]   = idx.day_of_year.astype(np.int16)

    # Cyclical (sin/cos) encodings so the model sees periodicity
    new_cols["hour_sin"]    = np.sin(2 * np.pi * idx.hour / 24).astype(np.float32)
    new_cols["hour_cos"]    = np.cos(2 * np.pi * idx.hour / 24).astype(np.float32)
    new_cols["dow_sin"]     = np.sin(2 * np.pi * idx.dayofweek / 7).astype(np.float32)
    new_cols["dow_cos"]     = np.cos(2 * np.pi * idx.dayofweek / 7).astype(np.float32)
    new_cols["month_sin"]   = np.sin(2 * np.pi * (idx.month - 1) / 12).astype(np.float32)
    new_cols["month_cos"]   = np.cos(2 * np.pi * (idx.month - 1) / 12).astype(np.float32)

    # Binary flags
    is_weekend = (idx.dayofweek >= 5).astype(np.float32)
    is_holiday = np.array(
        [d.date() in region_holidays for d in idx], dtype=np.float32
    )
    new_cols["is_weekend"]  = is_weekend
    new_cols["is_holiday"]  = is_holiday
    # Peak (17–20 h) and off-peak (< 7 h or ≥ 21 h) periods
    new_cols["is_peak"]     = ((idx.hour >= 17) & (idx.hour <= 20)).astype(np.float32)
    new_cols["is_shoulder"] = ((idx.hour >= 7)  & (idx.hour < 17)).astype(np.float32)
    new_cols["is_off_peak"] = ((idx.hour < 7)   | (idx.hour >= 21)).astype(np.float32)

    # Combined weekend/holiday flag — demand behaviour is quite different
    new_cols["is_offday"]   = np.maximum(is_weekend, is_holiday).astype(np.float32)

    return pd.DataFrame(new_cols, index=idx)

new_df = _add_time_features(df_base)

df = pd.concat([df, new_df], axis=1)

new_df[:10]



,hour,dayofweek,month,dayofyear,hour_sin,hour_cos,dow_sin,dow_cos,month_sin,month_cos,is_weekend,is_holiday,is_peak,is_shoulder,is_off_peak,is_offday
Date,,,,,,,,,,,,,,,,
2018-01-01 00:05:00,0,0,1,1,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0
2018-01-01 00:10:00,0,0,1,1,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0
2018-01-01 00:15:00,0,0,1,1,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0
2018-01-01 00:20:00,0,0,1,1,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0
2018-01-01 00:25:00,0,0,1,1,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0
2018-01-01 00:30:00,0,0,1,1,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0
2018-01-01 00:35:00,0,0,1,1,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0
2018-01-01 00:40:00,0,0,1,1,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0
2018-01-01 00:45:00,0,0,1,1,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0


In [6]:
def _add_lag_features(df: pd.DataFrame) -> pd.DataFrame:
    anchor_offsets = (-HALF_HOUR, 0, HALF_HOUR)
    LAG_INTERVALS = sorted(set([
        # Intrahour / intraday history.
        1, 2, 3, HALF_HOUR, PER_HOUR, 2 * PER_HOUR,
        4 * PER_HOUR, 6 * PER_HOUR, 12 * PER_HOUR,
        # Calendar-anchored history with a ±30-minute spread.
        *[PER_DAY + o for o in anchor_offsets],
        *[2 * PER_DAY + o for o in anchor_offsets],
        *[3 * PER_DAY + o for o in anchor_offsets],
        *[4 * PER_DAY + o for o in anchor_offsets],
        *[PER_WEEK + o for o in anchor_offsets],
        *[2 * PER_WEEK + o for o in anchor_offsets],
    ] + list(range(13 * PER_HOUR, 35 * PER_HOUR + 1, PER_HOUR))))

    new_cols = {}
    for col in df_core_columns:
        for lag in LAG_INTERVALS:
            new_cols[f"{col}_lag_{lag}"] = df[col].shift(lag).astype(np.float32)

    return pd.DataFrame(new_cols, index=df.index)

new_df = _add_lag_features(df_base)

df = pd.concat([df, new_df], axis=1)

new_df[:10]



,nsw_price_lag_1,nsw_price_lag_2,nsw_price_lag_3,nsw_price_lag_6,nsw_price_lag_12,nsw_price_lag_24,nsw_price_lag_48,nsw_price_lag_72,nsw_price_lag_144,nsw_price_lag_156,...,vic_price_lag_870,vic_price_lag_1146,vic_price_lag_1152,vic_price_lag_1158,vic_price_lag_2010,vic_price_lag_2016,vic_price_lag_2022,vic_price_lag_4026,vic_price_lag_4032,vic_price_lag_4038
Date,,,,,,,,,,,,,,,,,,,,,
2018-01-01 00:05:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:10:00,90.209999,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:15:00,95.540657,90.209999,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:20:00,96.293869,95.540657,90.209999,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:25:00,92.500000,96.293869,95.540657,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:30:00,92.499901,92.500000,96.293869,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:35:00,84.092339,92.499901,92.500000,90.209999,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:40:00,92.449867,84.092339,92.499901,95.540657,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:45:00,95.790154,92.449867,84.092339,96.293869,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
def _add_long_range_features(df: pd.DataFrame) -> pd.DataFrame:

    new_cols = {}

    for col in df_core_columns:

        ap   = np.arcsinh(df[col] / 100)
        BASE = PER_YEAR

        # --- Annual price lags (same period last year ± spread) ---
        for offset in range(-PER_HOUR, PER_HOUR + 1, HALF_HOUR):
            lag = BASE + offset
            new_cols[f"{col}_lag_annual_{'+' if offset >= 0 else ''}{offset}"] = (
                df[col].shift(lag).astype(np.float32)
            )
            new_cols[f"{col}_asinh_lag_annual_{'+' if offset >= 0 else ''}{offset}"] = (
                ap.shift(lag).astype(np.float32)
            )

        p_annual = df[col].shift(BASE)
        annual_context = 2 * PER_DAY
        min_context = 12 * PER_HOUR
        new_cols[f"{col}_annual_rmean_{annual_context}"] = p_annual.rolling(annual_context, min_periods=min_context).mean().astype(np.float32)
        new_cols[f"{col}_annual_rmax_{annual_context}"] = p_annual.rolling(annual_context, min_periods=min_context).max().astype(np.float32)
        new_cols[f"{col}_annual_rstd_{annual_context}"] = p_annual.rolling(annual_context, min_periods=min_context).std().astype(np.float32)

        # Spike count over the same two-day context last year.
        new_cols[f"{col}_annual_spike_{annual_context}"] = (p_annual >= 300).rolling(annual_context, min_periods=min_context).sum().astype(np.float32)

        # Year-on-year price change (current vs same period last year)
        new_cols[f"{col}_yoy_change"]  = (df[col] - df[col].shift(BASE)).astype(np.float32)
        new_cols[f"{col}_yoy_ratio"]   = (df[col] / (df[col].shift(BASE).abs() + 1)).clip(0, 20).astype(np.float32)

        # Two-week and six-week rolling stats are generated by
        # _add_rolling_features to avoid duplication.

    return pd.DataFrame(new_cols, index=df.index)

new_df = _add_long_range_features(df_base)

df = pd.concat([df, new_df], axis=1)

new_df[:10]



,nsw_price_lag_annual_-12,nsw_price_asinh_lag_annual_-12,nsw_price_lag_annual_-6,nsw_price_asinh_lag_annual_-6,nsw_price_lag_annual_+0,nsw_price_asinh_lag_annual_+0,nsw_price_lag_annual_+6,nsw_price_asinh_lag_annual_+6,nsw_price_lag_annual_+12,nsw_price_asinh_lag_annual_+12,...,vic_price_lag_annual_+6,vic_price_asinh_lag_annual_+6,vic_price_lag_annual_+12,vic_price_asinh_lag_annual_+12,vic_price_annual_rmean_576,vic_price_annual_rmax_576,vic_price_annual_rstd_576,vic_price_annual_spike_576,vic_price_yoy_change,vic_price_yoy_ratio
Date,,,,,,,,,,,,,,,,,,,,,
2018-01-01 00:05:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:10:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:15:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:20:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:25:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:35:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:40:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:45:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
def _add_rolling_features(df: pd.DataFrame) -> pd.DataFrame:
    """Rolling statistics computed on price available at each timestamp."""
    ROLLING_WINDOWS = sorted(set([
        4, 8, 2 * PER_HOUR, 4 * PER_HOUR, 12 * PER_HOUR,
        PER_DAY, 2 * PER_DAY, PER_WEEK, 2 * PER_WEEK, 6 * PER_WEEK,
    ]))

    new_cols = {}
    for col in df_core_columns:
        for w in ROLLING_WINDOWS:
            min_p = max(1, w // 2)
            rolled = df[col].rolling(w, min_periods=min_p)
            new_cols[f"{col}_rmean_{w}"] = rolled.mean().astype(np.float32)
            new_cols[f"{col}_rstd_{w}"]  = rolled.std().astype(np.float32)
            new_cols[f"{col}_rmax_{w}"]  = rolled.max().astype(np.float32)
            new_cols[f"{col}_rmin_{w}"]  = rolled.min().astype(np.float32)  

    return pd.DataFrame(new_cols, index=df.index)

new_df = _add_rolling_features(df_base)

df = pd.concat([df, new_df], axis=1)

new_df[:10]



,nsw_price_rmean_4,nsw_price_rstd_4,nsw_price_rmax_4,nsw_price_rmin_4,nsw_price_rmean_8,nsw_price_rstd_8,nsw_price_rmax_8,nsw_price_rmin_8,nsw_price_rmean_24,nsw_price_rstd_24,...,vic_price_rmax_2016,vic_price_rmin_2016,vic_price_rmean_4032,vic_price_rstd_4032,vic_price_rmax_4032,vic_price_rmin_4032,vic_price_rmean_12096,vic_price_rstd_12096,vic_price_rmax_12096,vic_price_rmin_12096
Date,,,,,,,,,,,,,,,,,,,,,
2018-01-01 00:05:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:10:00,92.875328,3.769344,95.540657,90.209999,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:15:00,94.014839,3.316542,96.293869,90.209999,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:20:00,93.636131,2.811877,96.293869,90.209999,93.636131,2.811877,96.293869,90.209999,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:25:00,94.208603,1.996805,96.293869,92.499901,93.408882,2.487608,96.293869,90.209999,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:30:00,91.346527,5.156233,96.293869,84.092339,91.856125,4.406461,96.293869,84.092339,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:35:00,90.385529,4.195525,92.500000,84.092339,91.940948,4.028785,96.293869,84.092339,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:40:00,91.208069,4.994664,95.790154,84.092339,92.422096,3.970444,96.293869,84.092339,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:45:00,90.832565,4.921089,95.790154,84.092339,92.520584,3.917148,96.293869,84.092339,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
def _add_regime_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Capture price-spike and volatility regime signals.
    All windows look backward only, so there is no future leakage.
    Daily/weekly rolling std is already produced by _add_rolling_features
    and is not repeated here.
    Returns only the new columns to avoid copying the full base frame.
    """
    new_cols = {}

    for col in df_core_columns:

        # Was there a high-price spike in the last 24 h?
        new_cols[f"{col}_spike_flag_{PER_DAY}"] = (df[col].rolling(PER_DAY).max() > 300).astype(np.float32)
        # Was there a negative price in the last 24 h?
        new_cols[f"{col}_neg_flag_{PER_DAY}"] = (df[col].rolling(PER_DAY).min() < 0).astype(np.float32)
        # Quantile context (recent price level relative to weekly range)
        new_cols[f"{col}_q90_{PER_WEEK}"] = df[col].rolling(PER_WEEK).quantile(0.90).astype(np.float32)
        new_cols[f"{col}_q10_{PER_WEEK}"] = df[col].rolling(PER_WEEK).quantile(0.10).astype(np.float32)
        new_cols[f"{col}_pct_rank_{PER_DAY}"] = (
            df[col].rolling(PER_DAY).rank(pct=True)
        ).astype(np.float32)

    return pd.DataFrame(new_cols, index=df.index)

new_df = _add_regime_features(df_base)

df = pd.concat([df, new_df], axis=1)

new_df[:10]



,nsw_price_spike_flag_288,nsw_price_neg_flag_288,nsw_price_q90_2016,nsw_price_q10_2016,nsw_price_pct_rank_288,qld_price_spike_flag_288,qld_price_neg_flag_288,qld_price_q90_2016,qld_price_q10_2016,qld_price_pct_rank_288,sa_price_spike_flag_288,sa_price_neg_flag_288,sa_price_q90_2016,sa_price_q10_2016,sa_price_pct_rank_288,vic_price_spike_flag_288,vic_price_neg_flag_288,vic_price_q90_2016,vic_price_q10_2016,vic_price_pct_rank_288
Date,,,,,,,,,,,,,,,,,,,,
2018-01-01 00:05:00,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN
2018-01-01 00:10:00,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN
2018-01-01 00:15:00,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN
2018-01-01 00:20:00,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN
2018-01-01 00:25:00,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN
2018-01-01 00:30:00,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN
2018-01-01 00:35:00,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN
2018-01-01 00:40:00,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN
2018-01-01 00:45:00,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN


In [10]:
def _add_time_since_spike_features(df: pd.DataFrame) -> pd.DataFrame:

    # Native feature-row duration in hours (5 minutes -> 1/12 hour).
    INTERVAL_H = variables.FEATURE_GRANULARITY_IN_MINUTES / 60.0
    # Maximum hours to report (cap at 2 weeks to avoid unbounded values early in series)
    MAX_HOURS  = 336.0  # 2 weeks

    new_cols = {}
    for col in df_core_columns:
        hours_since_by_thr = {}
        for threshold in [150, 300, 1000]:
            spike_flag = (df[col] >= threshold).astype(np.float32)

            positions       = pd.RangeIndex(len(df))
            last_spike_pos  = (
                pd.Series(np.where(spike_flag.values, positions, np.nan), index=df.index)
                .ffill()
                .fillna(-MAX_HOURS / INTERVAL_H)  # no prior spike seen → treat as max
            )
            intervals_since = (pd.Series(positions, index=df.index) - last_spike_pos).clip(upper=MAX_HOURS / INTERVAL_H)
            hours_since     = (intervals_since * INTERVAL_H).astype(np.float32)
            new_cols[f"{col}_hours_since_spike_{threshold}"] = hours_since
            hours_since_by_thr[threshold] = hours_since

            # Log-scale version reduces the numeric range for the tree learner
            new_cols[f"{col}_log1p_hours_since_spike_{threshold}"] = np.log1p(hours_since).astype(np.float32)

            # Binary flags: was there a spike at each lookback horizon?
            for lookback_h in [1, 6, 12, 24, 48, 168]:   # 1h, 6h, 12h, 24h, 48h, 1wk
                intervals = int(lookback_h / INTERVAL_H)
                new_cols[f"{col}_spike_{threshold}_last_{lookback_h}h"] = (
                    spike_flag.rolling(intervals, min_periods=1).max().astype(np.float32)
                )

        # Combined: hours since any of the three thresholds (summarises overall stress state)
        new_cols[f"{col}_hours_since_any_spike"] = pd.concat(
            [hours_since_by_thr[150], hours_since_by_thr[300], hours_since_by_thr[1000]], axis=1
        ).min(axis=1).astype(np.float32)

    return pd.DataFrame(new_cols, index=df.index)


new_df = _add_time_since_spike_features(df_base)

df = pd.concat([df, new_df], axis=1)

new_df[:10]



,nsw_price_hours_since_spike_150,nsw_price_log1p_hours_since_spike_150,nsw_price_spike_150_last_1h,nsw_price_spike_150_last_6h,nsw_price_spike_150_last_12h,nsw_price_spike_150_last_24h,nsw_price_spike_150_last_48h,nsw_price_spike_150_last_168h,nsw_price_hours_since_spike_300,nsw_price_log1p_hours_since_spike_300,...,vic_price_spike_300_last_168h,vic_price_hours_since_spike_1000,vic_price_log1p_hours_since_spike_1000,vic_price_spike_1000_last_1h,vic_price_spike_1000_last_6h,vic_price_spike_1000_last_12h,vic_price_spike_1000_last_24h,vic_price_spike_1000_last_48h,vic_price_spike_1000_last_168h,vic_price_hours_since_any_spike
Date,,,,,,,,,,,,,,,,,,,,,
2018-01-01 00:05:00,336.0,5.820083,0.0,0.0,0.0,0.0,0.0,0.0,336.0,5.820083,...,0.0,336.0,5.820083,0.0,0.0,0.0,0.0,0.0,0.0,336.0
2018-01-01 00:10:00,336.0,5.820083,0.0,0.0,0.0,0.0,0.0,0.0,336.0,5.820083,...,0.0,336.0,5.820083,0.0,0.0,0.0,0.0,0.0,0.0,336.0
2018-01-01 00:15:00,336.0,5.820083,0.0,0.0,0.0,0.0,0.0,0.0,336.0,5.820083,...,0.0,336.0,5.820083,0.0,0.0,0.0,0.0,0.0,0.0,336.0
2018-01-01 00:20:00,336.0,5.820083,0.0,0.0,0.0,0.0,0.0,0.0,336.0,5.820083,...,0.0,336.0,5.820083,0.0,0.0,0.0,0.0,0.0,0.0,336.0
2018-01-01 00:25:00,336.0,5.820083,0.0,0.0,0.0,0.0,0.0,0.0,336.0,5.820083,...,0.0,336.0,5.820083,0.0,0.0,0.0,0.0,0.0,0.0,336.0
2018-01-01 00:30:00,336.0,5.820083,0.0,0.0,0.0,0.0,0.0,0.0,336.0,5.820083,...,0.0,336.0,5.820083,0.0,0.0,0.0,0.0,0.0,0.0,336.0
2018-01-01 00:35:00,336.0,5.820083,0.0,0.0,0.0,0.0,0.0,0.0,336.0,5.820083,...,0.0,336.0,5.820083,0.0,0.0,0.0,0.0,0.0,0.0,336.0
2018-01-01 00:40:00,336.0,5.820083,0.0,0.0,0.0,0.0,0.0,0.0,336.0,5.820083,...,0.0,336.0,5.820083,0.0,0.0,0.0,0.0,0.0,0.0,336.0
2018-01-01 00:45:00,336.0,5.820083,0.0,0.0,0.0,0.0,0.0,0.0,336.0,5.820083,...,0.0,336.0,5.820083,0.0,0.0,0.0,0.0,0.0,0.0,336.0


In [11]:
def _add_spike_predictors(df: pd.DataFrame) -> pd.DataFrame:
    """
    Price momentum and spike-history features.
    All look-back only — zero leakage.
    The daily percentile rank is already produced by _add_regime_features
    and is not repeated here.
    Returns only the new columns to avoid copying the full base frame.
    """

    new_cols = {}

    for col in df_core_columns:

        # Price momentum — how fast prices are moving right now
        new_cols[f"{col}_mom_2h"] = df[col].diff(2 * PER_HOUR).astype(np.float32)
        new_cols[f"{col}_mom_6h"] = df[col].diff(6 * PER_HOUR).astype(np.float32)
        new_cols[f"{col}_mom_24h"] = df[col].diff(PER_DAY).astype(np.float32)

        # Price acceleration (second derivative) — is the current spike accelerating?
        new_cols[f"{col}_accel_2h"] = df[col].diff(2 * PER_HOUR).diff(2 * PER_HOUR).clip(-2000, 2000).astype(np.float32)
        new_cols[f"{col}_accel_6h"] = df[col].diff(6 * PER_HOUR).diff(6 * PER_HOUR).clip(-2000, 2000).astype(np.float32)

        # Spike and negative price counts in recent history
        new_cols[f"{col}_spike_count_{PER_DAY}"] = (df[col] >= 300).rolling(PER_DAY).sum().astype(np.float32)
        new_cols[f"{col}_spike_count_{PER_WEEK}"] = (df[col] >= 300).rolling(PER_WEEK).sum().astype(np.float32)
        new_cols[f"{col}_neg_count_{PER_DAY}"] = (df[col] < 0).rolling(PER_DAY).sum().astype(np.float32)

        # Spike intensity: cumulative spike energy in last 24h (not just count)
        SPIKE_THR = 150.0
        spike_excess = (df[col] - SPIKE_THR).clip(lower=0)
        new_cols[f"{col}_spike_intensity_{PER_DAY}"] = spike_excess.rolling(PER_DAY).sum().astype(np.float32)
        new_cols[f"{col}_spike_intensity_{PER_WEEK}"] = spike_excess.rolling(PER_WEEK).sum().astype(np.float32)

        # Price percentile rank over weekly window
        new_cols[f"{col}_pct_rank_{PER_WEEK}"] = df[col].rolling(PER_WEEK).rank(pct=True).astype(np.float32)

    return pd.DataFrame(new_cols, index=df.index)

new_df = _add_spike_predictors(df_base)

df = pd.concat([df, new_df], axis=1)

new_df[:10]



,nsw_price_mom_2h,nsw_price_mom_6h,nsw_price_mom_24h,nsw_price_accel_2h,nsw_price_accel_6h,nsw_price_spike_count_288,nsw_price_spike_count_2016,nsw_price_neg_count_288,nsw_price_spike_intensity_288,nsw_price_spike_intensity_2016,...,vic_price_mom_6h,vic_price_mom_24h,vic_price_accel_2h,vic_price_accel_6h,vic_price_spike_count_288,vic_price_spike_count_2016,vic_price_neg_count_288,vic_price_spike_intensity_288,vic_price_spike_intensity_2016,vic_price_pct_rank_2016
Date,,,,,,,,,,,,,,,,,,,,,
2018-01-01 00:05:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:10:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:15:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:20:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:25:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:35:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:40:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:45:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
def _add_region_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Pairwise spread features for all NEM region prices (NSW, QLD, VIC, SA).
    Captures interconnector pressure from any direction — useful for predicting any state.

    Lags, rolling stats, arcsinh lags, and spike counts are handled by their dedicated
    functions (_add_lag_features, _add_rolling_features, _add_arcsinh_price_lags,
    _add_spike_predictors) and are not repeated here.
    Returns only the new columns to avoid copying the full base frame.
    """

    new_cols: dict = {}

    for col in df_core_columns:
        # Pairwise spreads vs every other region — captures interconnector pressure
        for other_col in df_core_columns:
            if other_col == col:
                continue
            spread = (df[col] - df[other_col]).astype(np.float32)
            new_cols[f"{col}_vs_{other_col}_spread"]      = spread
            new_cols[f"{col}_vs_{other_col}_spread_lag1"] = spread.shift(1).astype(np.float32)

    # Multi-region spike co-occurrence: all regions elevated simultaneously
    regions_present = [c for c in df_core_columns if c in df.columns]
    if len(regions_present) >= 2:
        flags = pd.concat([(df[c] >= 150).astype(np.float32) for c in regions_present], axis=1)
        new_cols["multi_region_spike"] = flags.min(axis=1)  # 1 only if ALL elevated
        new_cols["region_spike_count"] = flags.sum(axis=1).astype(np.float32)  # 0–4

    return pd.DataFrame(new_cols, index=df.index)

new_df = _add_region_features(df_base)

df = pd.concat([df, new_df], axis=1)

new_df[:10]



,nsw_price_vs_qld_price_spread,nsw_price_vs_qld_price_spread_lag1,nsw_price_vs_sa_price_spread,nsw_price_vs_sa_price_spread_lag1,nsw_price_vs_vic_price_spread,nsw_price_vs_vic_price_spread_lag1,qld_price_vs_nsw_price_spread,qld_price_vs_nsw_price_spread_lag1,qld_price_vs_sa_price_spread,qld_price_vs_sa_price_spread_lag1,...,sa_price_vs_vic_price_spread,sa_price_vs_vic_price_spread_lag1,vic_price_vs_nsw_price_spread,vic_price_vs_nsw_price_spread_lag1,vic_price_vs_qld_price_spread,vic_price_vs_qld_price_spread_lag1,vic_price_vs_sa_price_spread,vic_price_vs_sa_price_spread_lag1,multi_region_spike,region_spike_count
Date,,,,,,,,,,,,,,,,,,,,,
2018-01-01 00:05:00,9.939713,NaN,-13.368568,NaN,-0.399643,NaN,-9.939713,NaN,-23.308281,NaN,...,12.968925,NaN,0.399643,NaN,10.339355,NaN,-12.968925,NaN,0.0,0.0
2018-01-01 00:10:00,10.440590,9.939713,-16.276489,-13.368568,-1.196190,-0.399643,-10.440590,-9.939713,-26.717079,-23.308281,...,15.080299,12.968925,1.196190,0.399643,11.636780,10.339355,-15.080299,-12.968925,0.0,0.0
2018-01-01 00:15:00,11.193871,10.440590,-17.409035,-16.276489,-1.050522,-1.196190,-11.193871,-10.440590,-28.602905,-26.717079,...,16.358513,15.080299,1.050522,1.196190,12.244392,11.636780,-16.358513,-15.080299,0.0,0.0
2018-01-01 00:20:00,10.771660,11.193871,-14.251778,-17.409035,-0.324448,-1.050522,-10.771660,-11.193871,-25.023438,-28.602905,...,13.927330,16.358513,0.324448,1.050522,11.096107,12.244392,-13.927330,-16.358513,0.0,0.0
2018-01-01 00:25:00,10.790970,10.771660,-16.031982,-14.251778,-0.333229,-0.324448,-10.790970,-10.771660,-26.822952,-25.023438,...,15.698753,13.927330,0.333229,0.324448,11.124199,11.096107,-15.698753,-13.927330,0.0,0.0
2018-01-01 00:30:00,10.362312,10.790970,-14.567451,-16.031982,-0.296669,-0.333229,-10.362312,-10.790970,-24.929764,-26.822952,...,14.270782,15.698753,0.296669,0.333229,10.658981,11.124199,-14.270782,-15.698753,0.0,0.0
2018-01-01 00:35:00,11.361214,10.362312,-12.550186,-14.567451,1.100349,-0.296669,-11.361214,-10.362312,-23.911400,-24.929764,...,13.650536,14.270782,-1.100349,0.296669,10.260864,10.658981,-13.650536,-14.270782,0.0,0.0
2018-01-01 00:40:00,11.190193,11.361214,-13.312645,-12.550186,1.875237,1.100349,-11.190193,-11.361214,-24.502838,-23.911400,...,15.187881,13.650536,-1.875237,-1.100349,9.314957,10.260864,-15.187881,-13.650536,0.0,0.0
2018-01-01 00:45:00,11.497093,11.190193,-14.002144,-13.312645,1.185738,1.875237,-11.497093,-11.190193,-25.499237,-24.502838,...,15.187881,15.187881,-1.185738,-1.875237,10.311356,9.314957,-15.187881,-15.187881,0.0,0.0


In [13]:
def _add_cross_feats(df: pd.DataFrame) -> pd.DataFrame:
    doy = df.index.day_of_year.astype(np.float32)
    new_cols = {}
    new_cols["doy_sin"] = np.sin(2 * np.pi * doy / 365.25).astype(np.float32)
    new_cols["doy_cos"] = np.cos(2 * np.pi * doy / 365.25).astype(np.float32)
    if "sa_price" in df.columns and "nsw_price" in df.columns:
        new_cols["sa_spread_live"] = np.arcsinh(
            (df["sa_price"].fillna(0.0).values - df["nsw_price"].fillna(0.0).values) / 100.0
        ).clip(-10, 10).astype(np.float32)
    return pd.DataFrame(new_cols, index=df.index)

new_df = _add_cross_feats(df_base)
df = pd.concat([df, new_df], axis=1)
new_df[:10]


,doy_sin,doy_cos,sa_spread_live
Date,,,
2018-01-01 00:05:00,0.017202,0.999852,0.133291
2018-01-01 00:10:00,0.017202,0.999852,0.162055
2018-01-01 00:15:00,0.017202,0.999852,0.173223
2018-01-01 00:20:00,0.017202,0.999852,0.142040
2018-01-01 00:25:00,0.017202,0.999852,0.159641
2018-01-01 00:30:00,0.017202,0.999852,0.145164
2018-01-01 00:35:00,0.017202,0.999852,0.125175
2018-01-01 00:40:00,0.017202,0.999852,0.132736
2018-01-01 00:45:00,0.017202,0.999852,0.139568


In [14]:
# Retain core columns: current dispatch prices are known at origin t, so keeping
# them as features is leakage-free (targets are strictly future, >= +30 min).
print("Total features:", df.shape[1])

df.to_parquet("../2_Features_build/Feature_data/1_dispatch_price.parquet")
df.shape

Total features: 681


(893664, 681)

In [15]:
# Free this kernel's memory so the next notebook has RAM to work with
# (clears data variables + returns freed heap to the OS).
release_memory()


[release_memory] cleared 14 variable(s); kernel rss 2.50G, 7.2G RAM free now
